# NeuralSpace Google Colab Bootstrap

This public notebook contains no secrets. Generate a one-time claim code in NeuralSpace, then run the cell below.

In [ ]:
from getpass import getpass
import requests

API_BASE = input("NeuralSpace public API URL (for example https://api.example.com/api/v1): ").strip().rstrip("/")
if not API_BASE.startswith("https://"):
    raise RuntimeError("NeuralSpace API URL must use HTTPS")

claim_code = getpass("Paste NeuralSpace claim code: ").strip()
response = requests.post(
    f"{API_BASE}/colab/claims/exchange",
    json={"claim_code": claim_code},
    timeout=30,
)
response.raise_for_status()

config = response.json()
RUNTIME_TOKEN = config["runtime_token"]
RUNTIME_HEADERS = {"Authorization": f"Bearer {RUNTIME_TOKEN}"}
claim_code = None
print("Connected to NeuralSpace")

In [ ]:
def heartbeat():
    response = requests.post(
        f"{API_BASE}/colab/runtime/heartbeat",
        headers=RUNTIME_HEADERS,
        timeout=30,
    )
    response.raise_for_status()
    return response.json()

heartbeat()
print(f"Datasets available: {len(config.get('datasets', []))}")

In [ ]:
run_response = requests.post(
    f"{API_BASE}/colab/runtime/runs",
    headers=RUNTIME_HEADERS,
    json={"name": "Frontend metrics and logs test"},
    timeout=30,
)
run_response.raise_for_status()
RUN_ID = run_response.json()["run_id"]

metrics_response = requests.post(
    f"{API_BASE}/colab/runtime/runs/{RUN_ID}/metrics",
    headers=RUNTIME_HEADERS,
    json={"values": {"loss": 0.42, "accuracy": 0.91}},
    timeout=30,
)
metrics_response.raise_for_status()

log_response = requests.post(
    f"{API_BASE}/colab/runtime/runs/{RUN_ID}/logs",
    headers=RUNTIME_HEADERS,
    json={"level": "INFO", "message": "Metrics and logs test completed"},
    timeout=30,
)
log_response.raise_for_status()
print("Sent metrics and logs:", RUN_ID)